# F1 Bronze — RESULTS — Multi-File Ingestion + Archive

### Flow
`ADLS Landing → Bronze Delta → Archive`

- Any number of JSON files can arrive at any time.
- No day-wise/date-wise processing logic.
- Bronze is append-only and preserves raw historical data.
- Files are moved to archive only after successful Bronze ingestion.
- `source_file` identifies the physical source file for each Bronze record.


## 1. Environment Configuration

Use the same `env` parameter pattern as the existing Formula 1 notebooks.

In [0]:
dbutils.widgets.text("env", "dev")

env = dbutils.widgets.get("env")

print("Environment:", env)

## 2. Define Catalog, Paths and Bronze Table

In [0]:
CATALOG = f"formula1_{env}"
BRONZE = "bronze"

landing_path = "/Volumes/formula1_dev/bronze/demo_files/latest_files_demo"
archive_path = "/Volumes/formula1_dev/bronze/demo_files/archive"

bronze_table = f"{CATALOG}.{BRONZE}.results_1"

print("Catalog       :", CATALOG)
print("Landing path  :", landing_path)
print("Archive path  :", archive_path)
print("Bronze table  :", bronze_table)

## 3. Create Bronze Schema

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE}")

print(f"Schema ready: {CATALOG}.{BRONZE}")

## 4. Define Source Schema

Explicit schema is used instead of schema inference.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    FloatType
)

results_schema = StructType([
    StructField("resultId", IntegerType(), False),
    StructField("raceId", IntegerType(), True),
    StructField("driverId", IntegerType(), True),
    StructField("constructorId", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("grid", IntegerType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("positionOrder", IntegerType(), True),
    StructField("points", FloatType(), True),
    StructField("laps", IntegerType(), True),
    StructField("time", StringType(), True),
    StructField("milliseconds", IntegerType(), True),
    StructField("fastestLap", IntegerType(), True),
    StructField("rank", IntegerType(), True),
    StructField("fastestLapTime", StringType(), True),
    StructField("fastestLapSpeed", FloatType(), True),
    StructField("statusId", StringType(), True)
])

## 5. Discover Files Currently in Landing

This is deliberately file-based rather than day-based. If 2, 10 or 100 files are present, all are picked up in this run.

In [0]:
files = dbutils.fs.ls(landing_path)

landing_files = [
    file.path
    for file in files
    if file.name.lower().endswith(".json")
]

print("Number of JSON files found:", len(landing_files))

for file in landing_files:
    print(file)

## 6. Stop Gracefully When No Files Are Available

In [0]:
if len(landing_files) == 0:
    print("No JSON files found in landing. Nothing to process.")
    input_file_name = []
else:
    input_file_name = [file.rstrip("/").split("/")[-1] for file in landing_files]
    print(f"{len(landing_files)} file(s) found. Starting ingestion...")
    for file in input_file_name:
        print(file)

In [0]:
print(landing_files)
print(input_file_name)

## 7. Read All Available Landing Files

In [0]:
if len(landing_files) > 0:
    results_df = (
        spark.read
        .schema(results_schema)
        .json(landing_files)
    )

    source_count = results_df.count()
    print("Source rows:", source_count)
    display(results_df.limit(20))

## 8. Add Bronze Metadata and Rename Columns

`source_file` tells us which physical file produced each record.
`ingestion_timestamp` is the Bronze arrival/ingestion timestamp used later by Silver as the watermark.

In [0]:
if len(landing_files) > 0:
    results_bronze = (
        results_df
        .withColumnRenamed("resultId", "result_id")
        .withColumnRenamed("raceId", "race_id")
        .withColumnRenamed("driverId", "driver_id")
        .withColumnRenamed("constructorId", "constructor_id")
        .withColumnRenamed("positionText", "position_text")
        .withColumnRenamed("positionOrder", "position_order")
        .withColumnRenamed("fastestLap", "fastest_lap")
        .withColumnRenamed("fastestLapTime", "fastest_lap_time")
        .withColumnRenamed("fastestLapSpeed", "fastest_lap_speed")
        .withColumn("source_file", F.col("_metadata.file_path"))
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("ingestion_date", F.current_date())
    )

    display(results_bronze)

## 9. Show File-Level Record Counts

Useful for validating that every input file contributed records.

In [0]:
if len(landing_files) > 0:
    display(
        results_bronze
        .groupBy("source_file")
        .count()
        .orderBy("source_file")
    )

## 10. Write to Bronze Delta

Use `append` because Bronze is the raw historical/source-of-truth layer. Do not overwrite the existing Bronze table.

In [0]:
if len(landing_files) > 0:
    (
        results_bronze
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(bronze_table)
    )

    print("Bronze ingestion completed successfully.")

## 11. Validate Bronze After Write

In [0]:
if len(landing_files) > 0:
    bronze_df = spark.table(bronze_table)

    print("Bronze total rows:", bronze_df.count())

    display(
        bronze_df
        .orderBy(F.desc("ingestion_timestamp"))
        # .limit(20)
    )

## 12. Archive Successfully Processed Files

This step runs only after the Bronze write succeeds. Therefore, a failed Bronze write leaves the source files in landing so they can be retried.

In [0]:
if len(landing_files) > 0:
    dbutils.fs.mkdirs(archive_path)

    for file in landing_files:
        file_name = file.rstrip("/").split("/")[-1]
        archive_file = f"{archive_path}/{file_name}"

        dbutils.fs.mv(file, archive_file)

        print(f"Archived: {file_name}")

    print("All successfully processed files moved to archive.")

## 13. Verify Landing Is Empty / Contains Only Unprocessed Files

In [0]:
remaining_files = [
    file.name
    for file in dbutils.fs.ls(landing_path)
    if file.name.lower().endswith(".json")
]

print("JSON files remaining in landing:", len(remaining_files))

for file in remaining_files:
    print(file)

## 14. Verify Archive


In [0]:
archived_files = [
    file.name
    for file in dbutils.fs.ls(archive_path)
    if file.name.lower().endswith(".json")
]

print("JSON files in archive:", len(archived_files))

for file in archived_files:
    print(file)

## 15. Final Bronze Summary

The Bronze layer now contains the raw historical records from all successfully ingested files. Silver can use `MAX(ingestion_timestamp)` as its watermark and perform incremental processing + MERGE.

In [0]:
if spark.catalog.tableExists(bronze_table):
    final_bronze = spark.table(bronze_table)

    print("====================================")
    print("BRONZE INGESTION SUMMARY")
    print("====================================")
    print("Bronze table :", bronze_table)
    print("Total rows   :", final_bronze.count())
    print("Total files  :", final_bronze.select("source_file").distinct().count())
    print("====================================")

    display(
        final_bronze
        .groupBy("source_file")
        .count()
        .orderBy(F.desc("count"))
    )